In [1]:
import pandas as pd

df = pd.read_csv("data/song_data.csv")

# Check the size of the dataset: how many songs (rows) and attributes (columns)?
print(df.shape)

# Check the data type of each column (int, float, string, etc.)
print(df.dtypes)

# Count missing values per column — any column with > 0 needs attention
print(df.isnull().sum())

# Get basic statistics: min, max, mean, std for all numeric columns
df.describe()

(18835, 15)
song_name            object
song_popularity       int64
song_duration_ms      int64
acousticness        float64
danceability        float64
energy              float64
instrumentalness    float64
key                   int64
liveness            float64
loudness            float64
audio_mode            int64
speechiness         float64
tempo               float64
time_signature        int64
audio_valence       float64
dtype: object
song_name           0
song_popularity     0
song_duration_ms    0
acousticness        0
danceability        0
energy              0
instrumentalness    0
key                 0
liveness            0
loudness            0
audio_mode          0
speechiness         0
tempo               0
time_signature      0
audio_valence       0
dtype: int64


,song_popularity,song_duration_ms,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,audio_mode,speechiness,tempo,time_signature,audio_valence
count,18835.000000,1.883500e+04,18835.000000,18835.000000,18835.000000,18835.000000,18835.000000,18835.000000,18835.000000,18835.000000,18835.000000,18835.000000,18835.000000,18835.000000
mean,52.991877,2.182116e+05,0.258539,0.633348,0.644995,0.078008,5.289196,0.179650,-7.447435,0.628139,0.102099,121.073154,3.959119,0.527967
std,21.905654,5.988754e+04,0.288719,0.156723,0.214101,0.221591,3.614595,0.143984,3.827831,0.483314,0.104378,28.714456,0.298533,0.244632
min,0.000000,1.200000e+04,0.000001,0.000000,0.001070,0.000000,0.000000,0.010900,-38.768000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,1.843395e+05,0.024100,0.533000,0.510000,0.000000,2.000000,0.092900,-9.044000,0.000000,0.037800,98.368000,4.000000,0.335000
50%,56.000000,2.113060e+05,0.132000,0.645000,0.674000,0.000011,5.000000,0.122000,-6.555000,1.000000,0.055500,120.013000,4.000000,0.527000
75%,69.000000,2.428440e+05,0.424000,0.748000,0.815000,0.002570,8.000000,0.221000,-4.908000,1.000000,0.119000,139.931000,4.000000,0.725000
max,100.000000,1.799346e+06,0.996000,0.987000,0.999000,0.997000,11.000000,0.986000,1.585000,1.000000,0.941000,242.318000,5.000000,0.984000


In [2]:
import pandas as pd

def load_and_clean():
    df = pd.read_csv("data/song_data.csv")

    # Drop rows missing critical columns — even though this dataset has none,
    # this is good defensive programming practice for any future data updates
    df = df.dropna(subset=["song_name", "song_popularity"])

    # Remove duplicate songs by name, keeping only the first occurrence
    df = df.drop_duplicates(subset=["song_name"], keep="first")

    # Remove songs with tempo = 0, which is physically impossible and signals bad data
    df = df[df["tempo"] > 0]

    # Remove songs shorter than 30 seconds (30,000 ms) — likely data errors or sound effects
    df = df[df["song_duration_ms"] >= 30000]

    # Remove songs longer than 10 minutes (600,000 ms) — likely live versions or podcasts
    # that would skew the duration analysis
    df = df[df["song_duration_ms"] <= 600000]

    # Convert duration from milliseconds to minutes for easier human interpretation
    df["duration_min"] = (df["song_duration_ms"] / 60000).round(2)

    # Create a categorical popularity column by dividing scores into 4 meaningful groups
    # This is basic feature engineering — turning a continuous number into a useful label
    # pd.cut() assigns each song to a bin based on its popularity score
    df["popularity_group"] = pd.cut(
        df["song_popularity"],
        bins=[0, 30, 60, 80, 100],
        labels=["Low (0–30)", "Medium (31–60)", "High (61–80)", "Viral (81–100)"]
    )

    # Reset the index after filtering so row numbers are clean and sequential (0, 1, 2, ...)
    # Without this, deleted rows leave "gaps" in the index which can cause bugs later
    df = df.reset_index(drop=True)

    # Print a quick summary to confirm the cleaning worked as expected
    print(f"✅ Clean dataset: {len(df):,} songs")
    print(f"   Average popularity: {df['song_popularity'].mean():.1f}")
    print(f"   Viral songs (>80): {(df['song_popularity'] > 80).sum():,}")

    return df

In [3]:
df = load_and_clean()

# Assert that no song has an impossible tempo value after cleaning
assert df["tempo"].min() > 0

# Assert that all songs fall within the expected duration range (0.5 to 10 minutes)
assert df["duration_min"].between(0.5, 10).all()

# Check how many songs fall into each popularity group
# This helps you understand the distribution before building any charts
print(df["popularity_group"].value_counts())

✅ Clean dataset: 13,051 songs
   Average popularity: 48.5
   Viral songs (>80): 354
popularity_group
Medium (31–60)    6787
High (61–80)      3550
Low (0–30)        2146
Viral (81–100)     354
Name: count, dtype: int64


In [1]:
import sys
sys.path.append("..")  # allow notebook to find the src/ folder

from src.load_data import load_and_clean
from src.queries import (
    feature_comparison_by_group,
    valence_vs_popularity,
    duration_vs_popularity,
    correlation_with_popularity
)

# Load clean data first
df = load_and_clean()

# Test each function one by one
print("=== Q1: Feature comparison by popularity group ===")
print(feature_comparison_by_group(df))

print("\n=== Q2: Happy vs Sad songs ===")
print(valence_vs_popularity(df))

print("\n=== Q3: Duration vs Popularity ===")
print(duration_vs_popularity(df))

print("\n=== Q4: Correlation with popularity ===")
print(correlation_with_popularity(df))

✅ Clean dataset: 13,051 songs
   Average popularity: 48.5
   Viral songs (>80): 354
=== Q1: Feature comparison by popularity group ===
  popularity_group  danceability  energy  acousticness  speechiness  liveness  \
0       Low (0–30)         0.616   0.643         0.293        0.098     0.188   
1   Medium (31–60)         0.621   0.627         0.294        0.101     0.180   
2     High (61–80)         0.630   0.644         0.251        0.099     0.177   
3   Viral (81–100)         0.699   0.666         0.185        0.118     0.163   

   audio_valence    tempo  
0          0.553  122.468  
1          0.527  120.926  
2          0.522  120.481  
3          0.489  122.414  

=== Q2: Happy vs Sad songs ===
                    mood  avg_popularity  song_count  median_popularity
0  Happy (valence ≥ 0.5)           47.77        7044               51.0
1    Sad (valence < 0.5)           49.38        6007               52.0

=== Q3: Duration vs Popularity ===
  duration_bracket  avg_popularity 

In [2]:
# Re-import để cập nhật các hàm mới thêm vào queries.py
import importlib
import src.queries
importlib.reload(src.queries)

from src.queries import (
    loudness_vs_popularity,
    liveness_vs_popularity,
    mode_vs_popularity,
    emotional_quadrant_analysis,
    song_profile_analysis
)

print("=== Q5: Loudness vs Popularity ===")
print(loudness_vs_popularity(df))

print("\n=== Q6: Live vs Studio Recordings ===")
print(liveness_vs_popularity(df))

print("\n=== Q7: Major vs Minor Key ===")
print(mode_vs_popularity(df))

print("\n=== Q8: Emotional Quadrants ===")
print(emotional_quadrant_analysis(df))

print("\n=== Q9: Song Profiles ===")
print(song_profile_analysis(df))

=== Q5: Loudness vs Popularity ===
          loudness_bracket  avg_popularity  song_count
0     Very Quiet (< -20dB)           52.48         250
1  Moderate (-20 to -10dB)           45.58        2616
2       Loud (-10 to -5dB)           48.46        6989
3       Very Loud (> -5dB)           50.72        3189

=== Q6: Live vs Studio Recordings ===
                      recording_type  avg_popularity  song_count  \
0    Live Recording (liveness > 0.8)           43.57          93   
1  Studio Recording (liveness ≤ 0.8)           48.55       12958   

   median_popularity  
0               49.0  
1               51.0  

=== Q7: Major vs Minor Key ===
         key_mode  avg_popularity  song_count  median_popularity
0  Major (bright)           48.51        8259               52.0
1    Minor (dark)           48.51        4792               51.0

=== Q8: Emotional Quadrants ===
      emotional_zone  avg_popularity  song_count  median_popularity
3         Sad & Slow           49.51        2268 